# Day 7: Backend Python: typing, dataclasses, SQLite, mocking and FastAPI (Solutions)

**Python for Everybody:** Course 4 *Using Databases with Python* (SQLite with `sqlite3`) and Course 5 Capstone ideas: take data, store it, serve it. Typing, dataclasses, mocking and FastAPI are beyond the course, so they use the official docs and fastapi.tiangolo.com.

**Why it matters for the JD:** The JD is 'Python and FastAPI' backend services that expose data securely, with automated tests. This day is the closest to the real job: validate data, talk to a database safely, write testable code, and serve a small API.

**Time:** about 100 minutes · 7 exercises

These are reference solutions. Try the exercises notebook first, then compare. Every cell here passes its check.

In [ ]:
# Run this cell first. It gives you the check() helper used below.
_results = {}

def check(name, test):
    """Run a test function and print a pass/fail line with a hint."""
    try:
        test()
    except AssertionError as e:
        _results[name] = False
        print(f"❌ {name}: {e}")
    except Exception as e:
        _results[name] = False
        print(f"❌ {name}: {type(e).__name__}: {e}")
    else:
        _results[name] = True
        print(f"✅ {name}")

def raises(exc, fn, *args, **kwargs):
    """True if fn(*args, **kwargs) raises exc."""
    try:
        fn(*args, **kwargs)
    except exc:
        return True
    except Exception:
        return False
    return False

TOTAL = 7

def score():
    done = sum(_results.values())
    print(f"Score: {done}/{TOTAL}  " + ("🎉 all green!" if done == TOTAL else "keep going, re-run the cells you missed"))


This day needs FastAPI and httpx (used by FastAPI's test client). Run the next cell once. In VS Code or Jupyter this installs into the notebook's environment. You may need to restart the kernel afterwards.

In [ ]:
%pip install -q fastapi httpx
import re
import sqlite3
from dataclasses import dataclass
from unittest.mock import Mock

## Exercise 1: Product dataclass with type hints  `warm-up`

Write a `@dataclass` named `Product` with fields `id: int`, `name: str`, `price: float`, `in_stock: bool = True`. Then write `cheapest_in_stock(products)` that returns the in-stock product with the lowest price, or `None` if none are in stock.

In [ ]:
@dataclass
class Product:
    id: int
    name: str
    price: float
    in_stock: bool = True


def cheapest_in_stock(products: list[Product]) -> Product | None:
    candidates = [p for p in products if p.in_stock]
    return min(candidates, key=lambda p: p.price) if candidates else None

In [ ]:
def _test():
    a = Product(1, "Pen", 10.0)
    assert a == Product(1, "Pen", 10.0), "dataclasses compare field by field"
    assert a.in_stock is True, "in_stock should default to True"
    items = [Product(1, "Pen", 10.0), Product(2, "Book", 5.0, in_stock=False), Product(3, "Bag", 20.0)]
    assert cheapest_in_stock(items) == items[0], "the cheapest in-stock item is the Pen (the Book is out of stock)"
    assert cheapest_in_stock([items[1]]) is None
    assert cheapest_in_stock([]) is None

check('1. Product dataclass with type hints', _test)

## Exercise 2: parse_product(d): validate incoming data  `medium`

Turn a raw dict (like a JSON body) into a `Product`, validating it:
- `id`, `name`, `price` are required. If one is missing raise `ValueError("missing field: <name>")`
- convert `id` to `int`, `price` to `float`, and strip spaces from `name`
- a negative price raises `ValueError`
- `in_stock` is optional and defaults to `True`

FastAPI does this for you with Pydantic. Here you do it by hand so you understand it.

In [ ]:
def parse_product(d: dict) -> Product:
    for key in ("id", "name", "price"):
        if key not in d:
            raise ValueError(f"missing field: {key}")
    price = float(d["price"])
    if price < 0:
        raise ValueError("price must be >= 0")
    return Product(int(d["id"]), str(d["name"]).strip(), price, bool(d.get("in_stock", True)))

In [ ]:
def _test():
    assert parse_product({"id": "7", "name": " Pen ", "price": "12.5"}) == Product(7, "Pen", 12.5, True)
    assert parse_product({"id": 1, "name": "x", "price": 1, "in_stock": False}).in_stock is False
    try:
        parse_product({"id": 1, "name": "x"})
    except ValueError as e:
        assert "price" in str(e), "the message should name the missing field"
    else:
        raise AssertionError("a missing price should raise ValueError")
    assert raises(ValueError, parse_product, {"id": 1, "name": "x", "price": -5}), "negative price should raise ValueError"

check('2. parse_product(d): validate incoming data', _test)

## Exercise 3: sqlite3: init_db, add_task, list_tasks  `medium`

Using the built-in `sqlite3` module (already imported), write:
- `init_db(conn)`: create table `tasks(id INTEGER PRIMARY KEY AUTOINCREMENT, title TEXT NOT NULL, done INTEGER NOT NULL DEFAULT 0)` if it doesn't exist
- `add_task(conn, title)`: insert a task using a **parameterized query** (`?`), commit, and return the new id
- `list_tasks(conn, only_open=False)`: return a list of dicts `{"id", "title", "done"}` ordered by id; with `only_open=True` only tasks where `done = 0`

In [ ]:
def init_db(conn):
    conn.execute(
        "CREATE TABLE IF NOT EXISTS tasks ("
        "id INTEGER PRIMARY KEY AUTOINCREMENT, "
        "title TEXT NOT NULL, "
        "done INTEGER NOT NULL DEFAULT 0)"
    )
    conn.commit()


def add_task(conn, title):
    cursor = conn.execute("INSERT INTO tasks (title) VALUES (?)", (title,))
    conn.commit()
    return cursor.lastrowid


def list_tasks(conn, only_open=False):
    sql = "SELECT id, title, done FROM tasks"
    if only_open:
        sql += " WHERE done = 0"
    sql += " ORDER BY id"
    conn.row_factory = sqlite3.Row
    return [dict(row) for row in conn.execute(sql)]

In [ ]:
def _test():
    conn = sqlite3.connect(":memory:")
    init_db(conn)
    init_db(conn)   # running it twice must not fail
    assert add_task(conn, "a") == 1
    assert add_task(conn, "b") == 2
    assert list_tasks(conn) == [{"id": 1, "title": "a", "done": 0}, {"id": 2, "title": "b", "done": 0}]
    conn.execute("UPDATE tasks SET done = 1 WHERE id = 1")
    conn.commit()
    assert [t["id"] for t in list_tasks(conn, only_open=True)] == [2], "only_open should hide finished tasks"

check('3. sqlite3: init_db, add_task, list_tasks', _test)

## Exercise 4: SQL injection: write find_tasks_safe  `interview`

`find_tasks_unsafe` is given and **vulnerable**: it pastes user input into the SQL text. Write `find_tasks_safe(conn, title)` that returns the matching tasks (same dict format as before) using a parameterized query.

The check shows the attack working on the unsafe function, and failing on yours.

In [ ]:
def find_tasks_unsafe(conn, title):
    # DON'T copy this. It builds SQL from user input.
    conn.row_factory = sqlite3.Row
    sql = f"SELECT id, title, done FROM tasks WHERE title = '{title}'"
    return [dict(row) for row in conn.execute(sql)]


def find_tasks_safe(conn, title):
    conn.row_factory = sqlite3.Row
    rows = conn.execute("SELECT id, title, done FROM tasks WHERE title = ?", (title,))
    return [dict(row) for row in rows]

In [ ]:
def _test():
    conn = sqlite3.connect(":memory:")
    init_db(conn)
    add_task(conn, "buy milk")
    add_task(conn, "call bob")
    evil = "x' OR '1'='1"
    assert len(find_tasks_unsafe(conn, evil)) == 2, "the attack works on the unsafe function (it leaks every row)"
    assert find_tasks_safe(conn, "buy milk") == [{"id": 1, "title": "buy milk", "done": 0}]
    assert find_tasks_safe(conn, evil) == [], "the safe version must treat the evil string as plain text"

check('4. SQL injection: write find_tasks_safe', _test)

## Exercise 5: get_user_name(fetch, user_id): testable code with Mock  `medium`

`fetch` is a function that loads a user dict from somewhere (an API, a database). Passing it in makes the code easy to test.

Return the user's `"name"`. If the dict has no name, or `fetch` raises `ConnectionError`, return `"unknown"`. Any **other** exception must not be hidden.

The check uses `unittest.mock.Mock` as a fake `fetch`.

In [ ]:
def get_user_name(fetch, user_id):
    try:
        return fetch(user_id)["name"]
    except (KeyError, ConnectionError):
        return "unknown"

In [ ]:
def _test():
    fake = Mock(return_value={"name": "Asha"})
    assert get_user_name(fake, 7) == "Asha"
    fake.assert_called_once_with(7)
    assert get_user_name(Mock(return_value={}), 1) == "unknown", "no name key"
    assert get_user_name(Mock(side_effect=ConnectionError), 1) == "unknown", "network failure"
    assert raises(ValueError, get_user_name, Mock(side_effect=ValueError), 1), "other errors must propagate"

check('5. get_user_name(fetch, user_id): testable code with Mock', _test)

## Exercise 6: FastAPI: build a tiny tasks API  `interview`

Build the three routes on the given `app`:
- `GET /health` returns `{"status": "ok"}`
- `POST /tasks` takes a JSON body `{"title": "..."}` (the Pydantic model `TaskIn` is given), stores a task in `TASKS`, returns status **201** and `{"id": 1, "title": ..., "done": false}`. Ids count 1, 2, 3...
- `GET /tasks/{task_id}` returns that task, or **404** with `{"detail": "Task not found"}`

FastAPI will reject a bad body (like `{}`) with 422 automatically. Run the check to test it with `TestClient`, with no server needed.

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI()
TASKS = {}


class TaskIn(BaseModel):
    title: str


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/tasks", status_code=201)
def create_task(task: TaskIn):
    task_id = len(TASKS) + 1
    TASKS[task_id] = {"id": task_id, "title": task.title, "done": False}
    return TASKS[task_id]


@app.get("/tasks/{task_id}")
def get_task(task_id: int):
    if task_id not in TASKS:
        raise HTTPException(status_code=404, detail="Task not found")
    return TASKS[task_id]

In [ ]:
def _test():
    from fastapi.testclient import TestClient

    TASKS.clear()
    client = TestClient(app)
    r = client.get("/health")
    assert r.status_code == 200 and r.json() == {"status": "ok"}, "GET /health should return {'status': 'ok'}"
    r = client.post("/tasks", json={"title": "write code"})
    assert r.status_code == 201, "POST /tasks should return 201"
    assert r.json() == {"id": 1, "title": "write code", "done": False}
    assert client.post("/tasks", json={"title": "test code"}).json()["id"] == 2
    assert client.get("/tasks/1").json()["title"] == "write code"
    missing = client.get("/tasks/99")
    assert missing.status_code == 404 and missing.json() == {"detail": "Task not found"}
    assert client.post("/tasks", json={}).status_code == 422, "an invalid body should give 422 automatically"

check('6. FastAPI: build a tiny tasks API', _test)

## Exercise 7: Write the test cases: slugify  `interview`

`slugify` is given. It turns a title into a URL-friendly slug. Your job is the **tests**, which the JD asks for: fill the list `cases` with at least **5** `(input, expected)` pairs that include:
- an empty string
- leading/trailing spaces
- repeated spaces inside
- punctuation (one of `! ? , . : ; '`)

Every case must pass `slugify`. Good engineers test the edges, not just the happy path.

In [ ]:
def slugify(title):
    words = re.findall(r"[a-z0-9]+", title.lower())
    return "-".join(words)


cases = [
    ("Hello World", "hello-world"),
    ("", ""),
    ("  padded  ", "padded"),
    ("many    spaces   here", "many-spaces-here"),
    ("Wow! Really?", "wow-really"),
    ("Python 3.12: What's new", "python-3-12-what-s-new"),
]

In [ ]:
def _test():
    assert len(cases) >= 5, "write at least 5 cases"
    assert all(slugify(i) == e for i, e in cases), "every case must pass slugify (check your expected values)"
    inputs = [i for i, _ in cases]
    assert "" in inputs, "include an empty string"
    assert any(i != i.strip() for i in inputs), "include leading or trailing spaces"
    assert any("  " in i.strip() for i in inputs), "include repeated spaces inside the text"
    assert any(any(ch in i for ch in "!?,.:;'") for i in inputs), "include punctuation"

check('7. Write the test cases: slugify', _test)

## Your score

In [ ]:
score()

## Say it out loud (interview questions)

Answer each one aloud in under 60 seconds, then compare.

**How do you prevent SQL injection?**

Never build SQL by concatenating or f-stringing user input. Use parameterized queries (`?` or `%s` placeholders) so the driver sends values separately from the SQL text. ORMs do this for you; also validate input and use least-privilege database accounts.

**Why use dependency injection or mocks in tests?**

Passing collaborators in (a `fetch` function, a DB connection) lets tests substitute fakes: fast, deterministic, no network. `unittest.mock` records calls so you can assert how the dependency was used.

**What does FastAPI give you over plain Flask-style code?**

Type-hint-driven validation and serialization via Pydantic, automatic 422 errors, auto-generated OpenAPI docs (`/docs`), dependency injection, and async support. Types are the contract.

**PUT vs PATCH? 401 vs 403?**

PUT replaces the whole resource; PATCH changes part of it. 401 means unauthenticated (we don't know who you are); 403 means authenticated but not allowed.

**What is a dataclass and when do you use one?**

A class decorator that generates `__init__`, `__repr__` and `__eq__` from annotated fields. Use it for simple data containers instead of hand-written boilerplate; `frozen=True` makes instances immutable.

